# 25 — Report Revision Pre-Adam v1

This notebook prepares revised report tables and wording after:

- caveat recoding;
- party-transition diagnostics;
- SDP campaign-performance validation.

It creates a pre-sharing report asset pack. It does not create the final designed report automatically.

In [1]:
from pathlib import Path
from datetime import datetime
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 240)

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
GEOGRAPHY_DIR = DATA_DIR / "geography"
DICTIONARY_DIR = DATA_DIR / "dictionaries"

for d in [PROCESSED_DIR, GEOGRAPHY_DIR, DICTIONARY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Processed:", PROCESSED_DIR)

Project: c:\Users\keena\Documents\Electoral_Tribes
Processed: c:\Users\keena\Documents\Electoral_Tribes\data\processed


In [2]:
OUTPUT_DIR = PROCESSED_DIR / "pre_adam_report_revision_v1"
TABLE_DIR = OUTPUT_DIR / "tables"
TEXT_DIR = OUTPUT_DIR / "draft_text"
MANIFEST_DIR = OUTPUT_DIR / "manifest"
for d in [OUTPUT_DIR, TABLE_DIR, TEXT_DIR, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("Output:", OUTPUT_DIR)

Output: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v1


In [3]:
def find_file(filename, search_dirs=None, required=True):
    if search_dirs is None:
        search_dirs = [
            PROCESSED_DIR / "target_review_pack_v1",
            PROCESSED_DIR / "target_model_v2",
            PROCESSED_DIR / "target_model_v1",
            PROCESSED_DIR / "report_assets_v1",
            PROCESSED_DIR / "election_results",
            PROCESSED_DIR,
            DATA_DIR / "raw" / "election_results",
            DATA_DIR / "raw",
            PROJECT_DIR,
            Path.cwd(),
        ]
    for folder in search_dirs:
        path = folder / filename
        if path.exists():
            return path
    # recursive fallback under processed and data/raw
    for root in [PROCESSED_DIR, DATA_DIR / "raw"]:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return sorted(matches, key=lambda p: p.stat().st_mtime, reverse=True)[0]
    if required:
        raise FileNotFoundError(f"Could not find {filename}")
    return None


def read_csv(filename, required=True):
    path = find_file(filename, required=required)
    if path is None:
        print("Optional file missing:", filename)
        return None, None
    df = pd.read_csv(path, low_memory=False)
    print(f"Loaded {filename}: {df.shape} from {path}")
    return df, path


def save_csv(df, path, index=False):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)
    print("Saved:", path, df.shape)
    return path


def boolish(s):
    return s.fillna(False).astype(str).str.strip().str.lower().isin(["true", "1", "yes", "y"])


def to_num(s):
    return pd.to_numeric(s, errors="coerce")


def safe_col(df, col, default=np.nan):
    if col in df.columns:
        return df[col]
    return pd.Series(default, index=df.index)


def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = x.replace("&", " and ")
    x = re.sub(r"[^a-z0-9]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()

## 25.1 Load revised model layers

In [4]:
revised, _ = read_csv("north_west_revised_consolidated_review_v1.csv", required=False)
if revised is None:
    revised, _ = read_csv("north_west_consolidated_target_review_v1.csv")

transition, _ = read_csv("north_west_party_transition_diagnostics_all_v1.csv", required=False)
sdp_profile, _ = read_csv("sdp_campaign_wards_profile_v1.csv", required=False)
sdp_count_check, _ = read_csv("sdp_candidate_count_check_v1.csv", required=False)

print("Revised review:", revised.shape)
print("Transition diagnostics:", None if transition is None else transition.shape)
print("SDP profile:", None if sdp_profile is None else sdp_profile.shape)

Loaded north_west_revised_consolidated_review_v1.csv: (825, 59) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1_revised_caveats\north_west_revised_consolidated_review_v1.csv
Loaded north_west_party_transition_diagnostics_all_v1.csv: (825, 22) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\party_transition_diagnostics_v1\north_west_party_transition_diagnostics_all_v1.csv
Loaded sdp_campaign_wards_profile_v1.csv: (171, 76) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v1\sdp_campaign_wards_profile_v1.csv
Loaded sdp_candidate_count_check_v1.csv: (6, 4) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v1\sdp_candidate_count_check_v1.csv
Revised review: (825, 59)
Transition diagnostics: (825, 22)
SDP profile: (171, 76)


## 25.2 Create revised report headline metrics

In [5]:
metrics = {
    "north_west_rows": len(revised),
    "high_confidence_rows": int((revised.get("report_confidence_band", pd.Series(index=revised.index)) == "High confidence").sum()),
    "medium_confidence_rows": int((revised.get("report_confidence_band", pd.Series(index=revised.index)) == "Medium confidence").sum()),
    "serious_caveat_rows": int((revised.get("report_confidence_band", pd.Series(index=revised.index)) == "Serious caveat / manual review").sum()),
    "clean_or_medium_opportunities": int(revised.get("revised_strategic_lane", pd.Series(index=revised.index)).isin(["Clean Opportunity", "Medium-Confidence Opportunity"]).sum()),
    "breakthrough_build_rows": int((revised.get("revised_strategic_lane", pd.Series(index=revised.index)) == "Breakthrough Build").sum()),
    "demographic_build_rows": int((revised.get("revised_strategic_lane", pd.Series(index=revised.index)) == "Long-Term Demographic Build").sum()),
}
if sdp_profile is not None:
    metrics["sdp_candidate_rows_loaded"] = len(sdp_profile)
    metrics["sdp_rows_matched_to_model"] = int(boolish(sdp_profile.get("matched_to_model", pd.Series(False, index=sdp_profile.index))).sum())
else:
    metrics["sdp_candidate_rows_loaded"] = 0
    metrics["sdp_rows_matched_to_model"] = 0

metrics_df = pd.DataFrame([metrics])
save_csv(metrics_df, TABLE_DIR / "pre_adam_headline_metrics_v1.csv")
display(metrics_df)

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v1\tables\pre_adam_headline_metrics_v1.csv (1, 9)


,north_west_rows,high_confidence_rows,medium_confidence_rows,serious_caveat_rows,clean_or_medium_opportunities,breakthrough_build_rows,demographic_build_rows,sdp_candidate_rows_loaded,sdp_rows_matched_to_model
0,825,0,0,825,0,0,0,171,107


## 25.3 Revised report tables

In [6]:
# High/medium/serious confidence tables.
for band, filename in [
    ("High confidence", "pre_adam_high_confidence_rows_v1.csv"),
    ("Medium confidence", "pre_adam_medium_confidence_rows_v1.csv"),
    ("Serious caveat / manual review", "pre_adam_serious_caveat_rows_v1.csv"),
]:
    if "report_confidence_band" in revised.columns:
        save_csv(revised[revised["report_confidence_band"].eq(band)].sort_values("initial_watchlist_score", ascending=False), TABLE_DIR / filename)

# Party transition tables.
if transition is not None:
    for score_col, filename in [
        ("conservative_transition_score", "pre_adam_conservative_transition_top_v1.csv"),
        ("labour_stronghold_breakthrough_score", "pre_adam_labour_stronghold_breakthrough_top_v1.csv"),
        ("reform_independent_disruption_score", "pre_adam_reform_independent_disruption_top_v1.csv"),
        ("green_ld_noncore_score", "pre_adam_green_ld_noncore_top_v1.csv"),
    ]:
        if score_col in transition.columns:
            save_csv(transition.sort_values(score_col, ascending=False).head(50), TABLE_DIR / filename)

# SDP validation tables.
if sdp_profile is not None:
    save_csv(sdp_profile.sort_values("sdp_vote_share", ascending=False), TABLE_DIR / "pre_adam_sdp_campaign_wards_profile_v1.csv")
    if "dominant_cluster_name" in sdp_profile.columns:
        sdp_by_tribe = sdp_profile.groupby("dominant_cluster_name", dropna=False).agg(
            sdp_rows=("candidate_name", "count"),
            mean_sdp_vote_share=("sdp_vote_share", "mean"),
            max_sdp_vote_share=("sdp_vote_share", "max"),
            total_sdp_votes=("sdp_votes", "sum"),
            mean_model_score=("initial_watchlist_score", "mean") if "initial_watchlist_score" in sdp_profile.columns else ("sdp_votes", "sum"),
        ).reset_index().sort_values("mean_sdp_vote_share", ascending=False)
        save_csv(sdp_by_tribe, TABLE_DIR / "pre_adam_sdp_performance_by_dominant_tribe_v1.csv")

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v1\tables\pre_adam_high_confidence_rows_v1.csv (0, 59)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v1\tables\pre_adam_medium_confidence_rows_v1.csv (0, 59)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v1\tables\pre_adam_serious_caveat_rows_v1.csv (825, 59)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v1\tables\pre_adam_conservative_transition_top_v1.csv (50, 22)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v1\tables\pre_adam_labour_stronghold_breakthrough_top_v1.csv (50, 22)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v1\tables\pre_adam_reform_independent_disruption_top_v1.csv (50, 22)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v1\tabl

## 25.4 Draft revised report language

These `.md` snippets are intended to be copied into the final report.

In [7]:
summary_text = f"""
# Revised model interpretation — pre-Adam draft

The current model should be described as a **Structural Breakthrough and Build Model**, not a seat-winning or council-control model.

It identifies wards where the demographic, electoral and political context appears favourable for one of four purposes:

1. near-term opportunity review;
2. medium-confidence opportunity review where evidence requires confirmation;
3. breakthrough-build work in apparently safe or entrenched-party wards;
4. longer-term demographic and organisational build.

The model does not yet predict vote share, councillor numbers, group formation, balance-of-power potential or council control. Those require a separate vote-share and seat-simulation layer.

Current North West headline after caveat recoding:

- High-confidence rows: {metrics['high_confidence_rows']}
- Medium-confidence rows: {metrics['medium_confidence_rows']}
- Serious caveat/manual-review rows: {metrics['serious_caveat_rows']}
- Clean or medium-confidence opportunities: {metrics['clean_or_medium_opportunities']}
- Breakthrough-build rows: {metrics['breakthrough_build_rows']}
- Long-term demographic-build rows: {metrics['demographic_build_rows']}
- SDP candidate rows loaded: {metrics['sdp_candidate_rows_loaded']}
- SDP candidate rows matched to model: {metrics['sdp_rows_matched_to_model']}

The key remaining limitation is the absence of a fully standardised SDP campaign-performance layer, especially for 2026. Once 2026 SDP candidate results are added and mapped, the model can begin testing whether the hypothesised SDP-aligned demographic terrain corresponds to actual SDP vote performance.
""".strip()

method_text = """
# Revised methodology note

The model has three distinct evidential layers.

First, the demographic layer groups Output Areas into seven politically neutral demographic tribes. These are not voters and should not be treated as party supporters. They are social geographies.

Second, the electoral layer overlays recent local-election evidence. This provides current party competition, fragmentation, margin, and political-openness indicators.

Third, the validation layer compares actual SDP campaign performance against the model. This layer is currently incomplete and should be treated as the next major dependency, especially once 2026 candidate results are standardised and mapped.

The current outputs should therefore be interpreted as structural opportunity evidence, not predictive election forecasts.
""".strip()

control_text = """
# Control-geography caveat

This model does not identify a council-control pathway. It identifies structurally favourable wards for breakthrough and organisational build. At this stage, the North West results suggest dispersed opportunity rather than a concentrated path to majority control in any single local authority.

A separate vote-share and seat-simulation model would be required to estimate ward wins, councillor numbers, minority influence, opposition group status, balance-of-power potential or council control.
""".strip()

for filename, text in [
    ("pre_adam_revised_model_interpretation.md", summary_text),
    ("pre_adam_revised_methodology_note.md", method_text),
    ("pre_adam_control_geography_caveat.md", control_text),
]:
    path = TEXT_DIR / filename
    path.write_text(text, encoding="utf-8")
    print("Saved:", path)

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v1\draft_text\pre_adam_revised_model_interpretation.md
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v1\draft_text\pre_adam_revised_methodology_note.md
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v1\draft_text\pre_adam_control_geography_caveat.md


## 25.5 Manifest

In [8]:
manifest_rows = []
for folder in [TABLE_DIR, TEXT_DIR]:
    for path in sorted(folder.glob("*")):
        manifest_rows.append({
            "filename": path.name,
            "folder": str(path.parent),
            "asset_type": "markdown_text" if path.suffix == ".md" else "csv_table",
            "created_at": datetime.now().isoformat(timespec="seconds"),
        })
manifest = pd.DataFrame(manifest_rows)
save_csv(manifest, MANIFEST_DIR / "pre_adam_report_revision_manifest_v1.csv")
display(manifest)

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v1\manifest\pre_adam_report_revision_manifest_v1.csv (13, 4)


,filename,folder,asset_type,created_at
0,pre_adam_conservative_transition_top_v1.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,csv_table,2026-05-28T13:51:47
1,pre_adam_green_ld_noncore_top_v1.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,csv_table,2026-05-28T13:51:47
2,pre_adam_headline_metrics_v1.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,csv_table,2026-05-28T13:51:47
3,pre_adam_high_confidence_rows_v1.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,csv_table,2026-05-28T13:51:47
4,pre_adam_labour_stronghold_breakthrough_top_v1...,c:\Users\keena\Documents\Electoral_Tribes\data...,csv_table,2026-05-28T13:51:47
5,pre_adam_medium_confidence_rows_v1.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,csv_table,2026-05-28T13:51:47
6,pre_adam_reform_independent_disruption_top_v1.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,csv_table,2026-05-28T13:51:47
7,pre_adam_sdp_campaign_wards_profile_v1.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,csv_table,2026-05-28T13:51:47
8,pre_adam_sdp_performance_by_dominant_tribe_v1.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,csv_table,2026-05-28T13:51:47
9,pre_adam_serious_caveat_rows_v1.csv,c:\Users\keena\Documents\Electoral_Tribes\data...,csv_table,2026-05-28T13:51:47


## 25.6 Next step

Once this notebook has run, revise the draft report by replacing the old caveat language with the new confidence-band language, adding the party-transition diagnostic section, and adding the SDP campaign-validation section once data is available.